In [ ]:

!python ../clean_codes_v2/run_inference_pipeline.py --config-file train_10_12.json --test 1 --Num 3 --fresh_run 1

Note: Test shapes are always saved as 99.npy
base_dir /mnt/home/project/cshukla.gitika/Mega_PartII/Mega_Runs_Pegasus/Data
... initialization complete.
If testing pipeline set: --fresh_run = 1
Running pipeline afresh
a.min(),a.max() 0.2291 0.6914
b.min(),b.max() 0.0628 0.4418
rprs.min(),rprs.max() 0.008919 0.415349
loc_cut_rprs =0.0833, high_cut_rprs = 0.1
pairs.shape (5711, 4)
rprs_grid (5711,) 5711
Number of LDC & Rp/Rs points is 5711
radius ratio (Rs/Rp) [11. 11. 11. ... 11. 11. 11.] 10.0 12.0
False
pairs[0:3,:] [[ 0.2291      0.39935255  0.08706948 11.        ]
 [ 0.23282823  0.39695547  0.0903067  11.        ]
 [ 0.23655645  0.39455839  0.09279795 11.        ]]
Figure(1000x500)
Figure(1000x500)
LDC giving negative intesnity (array([], dtype=int64),)
Generating 3 random closed Bezier shapes of size 38x38...
100%|████████████████████████████████████████████| 3/3 [00:00<00:00, 514.70it/s]

--- Output Summary (Filled Shapes) ---
Successfully generated and rasterized 3 FILLED shapes.
Sa

In [2]:
import sys
sys.path.append("/mnt/home/project/cshukla.gitika/Mega_PartII/clean_codes_v2")
from binary_classifier import *
from matplotlib.patches import Circle
import pandas as pd

In [ ]:

def binary_classification(N=None,snr=None,rsrp1=None,rsrp2=None,lc_dir=None):
    # Binary classification of 300 test shapes
    # 150 are orignally anomalous
    # 150 are circles
    # N = 99 for test shapes
    
    # snr_values = [10000,500,100]
    # N=99
    # rsrp1 = 2
    # rsrp2 = 5
    # snr = snr_values[2]
    
    #lc_dir = "/home/iit-t/Gitika/Github-Repositories/Abraham_Mega/Reanalysis_Git/Data_MegaPart1/"
    
    orig_filename = f"{lc_dir}{N}.npy"
    pred_filename = f"{lc_dir}{N}LC_rsrp{rsrp1}_rsrp{rsrp2}_processed_snr{snr}_PredctdShape.npy"
    
    orig_shapes = np.load(orig_filename)
    pred_shapes = np.load(pred_filename)
    
    #flux_cutI = 0.94
    flux_cutI = 0.93
    dist_cutII = 0.10
    deviation_estimator = 'flux'
    nproc = 20
    
    # original shapes
    binclas_orig,acc_orig,CI_orig,CII_orig,flags_str_orig,rad_fit_orig, fmaps_orig = batch_predict_shape(
        images=orig_shapes,deviation_estimator=deviation_estimator,
        cut1=flux_cutI, cut2=dist_cutII,
        num_cpus=nproc, show= False) 
    
    
    print('###############################################')
    print("Regarding Original Binary Mask")
    print(f'By eye no. of anomalous shapes = {150}, 50% of total')
    print(f'By CI-CII classification no. of anomalous shapes = {acc_orig*len(orig_shapes)}')
    print(f'Classification accuracy on original mask using threshold is = {round(acc_orig*100)}%')
    print('###############################################')
    
    
    # predicted shapes
    binclas_pred,acc_pred,CI_pred,CII_pred,flags_str_pred,rad_fit_pred, fmaps_pred = batch_predict_shape(
        images=pred_shapes,deviation_estimator=deviation_estimator,
        cut1=flux_cutI, cut2=dist_cutII,num_cpus=nproc, show= False)
    
    print('###############################################')
    print("Regarding Predicted Probability Mask")
    print(f'By CI-CII classification no. of anomalous shapes = {acc_pred*len(pred_shapes)}')
    print(f'Classification accuracy on predicted mask using threshold is = {round(acc_pred*100)}%')
    print('###############################################')
    
    
    print(f'The accuracy of predicted shapes should have been {round(acc_orig*100)}% but getting {round(acc_pred*100)}%')
    print('===============================================')
    
    print(f'Confusion Metrix Results')
    print('===============================================')
    
    savefigname = f'Mega_Part1_Figures/confusion_met_{N}_rsrp{rsrp1}_rsrp{rsrp2}_snr{snr}.png'
    results = estimate_ml_metrics(binclas_orig, binclas_pred, 
                                savefig=None)
    
    #save_dir="/home/iit-t/Gitika/Github-Repositories/Abraham_Mega/Reanalysis_Git/Data_MegaPart1/"
    np.savez(
        lc_dir+f"{N}_classification_params_rsrp{rsrp1}_rsrp{rsrp2}_snr{snr}.npz",
        binclass_orig=binclas_orig,     
        CI_orig=CI_orig,     
        CII_orig=CII_orig,  
        acc_orig=acc_orig,
        binclass_flag_orig=flags_str_orig,     
        circ_fit_orig=rad_fit_orig,              
        finalmaps_orig=fmaps_orig,  
        
        binclass_pred=binclas_pred,     
        CI_pred=CI_pred,     
        CII_pred=CII_pred, 
        acc_pred=acc_pred,
        binclass_flag_pred=flags_str_pred,     
        circ_fit_pred=rad_fit_pred,              
        finalmaps_pred=fmaps_pred,  

        ml_metric=results
    )
    return


def plot_maps_n_circle_fit(data_dir=None, N=None,snr=None,rsrp1=None,rsrp2=None,
                          ):

    # Plot the fitted circle over maps for orig shapes
    images = orig_shapes
    radius_fitted = rad_fit_orig
    # predictions_str = flags_str_orig
    # final_maps = fmaps_orig
    
    
    #save_dir="/home/iit-t/Gitika/Github-Repositories/Abraham_Mega/Reanalysis_Git/Data_MegaPart1/"    
    data = np.load(lc_dir+f"{N}_classification_params_rsrp{rsrp1}_rsrp{rsrp2}_snr{snr}.npz",allow_pickle=True)

    name = N

    # ----------------------------------------------------
    # plot originals
    images = data["finalmaps_orig"]
    radius_fitted = data["circ_fit_orig"]
    predictions_str = data["binclass_flag_orig"]
    final_maps = data["finalmaps_orig"]
    
    N = images.shape[0]
    nx,ny = images[0].shape
    print('N',N)
    
    images_per_row = min(10, N)  # If N < 10, show all in one row
    rows = (N + images_per_row - 1) // images_per_row  # Ceiling division
    cols = images_per_row
    
    # Create subplots
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    
      # Ensure axes is always 2D for uniform handling
    axes = np.atleast_2d(axes)
    
    for i in range(N):
        image = images[i,:,:]
        color = 'red'
        if predictions_str[i] == "Circle":
            color = 'cyan'
        best_cx,best_cy, best_radius = radius_fitted[i,0],radius_fitted[i,1],radius_fitted[i,2]
        circle = Circle((best_cx ,best_cy), best_radius, edgecolor=color, facecolor='none', linewidth=3)
    
      # Plot images
      #for idx in range(N):
        row = i // images_per_row
        col = i % images_per_row
        ax = axes[row, col]
        ax.imshow(final_maps[i],  cmap='inferno')
        #ax.plot(fitimg.x_circle, fitimg.y_circle, 'lime', label=f'Fitted Circle (r={fitimg.best_radius:.2f})', linewidth=1)
        #ax.contour(fitimg.image_padded, levels=[fitimg.threshold], colors='red', linewidths=2)
        ax.add_patch(circle)
    
        ax.set_title(f'{predictions_str[i], i}', fontsize=15,color=color)
        #ax.legend(loc='lower right', fontsize=6, frameon=True)
        ax.axis('off')
    
      # Hide any unused subplots
    for row in range(rows):
        for col in range(cols):
            if row * images_per_row + col >= N:
                axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.savefig(
        f'Mega_Part1_Figures/Diagnostic_Figures/{name}_orig_maps.png',
        dpi=100,
        bbox_inches='tight',
        pad_inches=0.2
        )
    
    plt.show()
    # ----------------------------------------------------
    images = data["finalmaps_pred"]
    radius_fitted = data["circ_fit_pred"]
    predictions_str = data["binclass_flag_pred"]
    final_maps = data["finalmaps_pred"]
    
    N = images.shape[0]
    nx,ny = images[0].shape
    print('N',N)
    
    images_per_row = min(10, N)  # If N < 10, show all in one row
    rows = (N + images_per_row - 1) // images_per_row  # Ceiling division
    cols = images_per_row
    
    # Create subplots
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    
      # Ensure axes is always 2D for uniform handling
    axes = np.atleast_2d(axes)
    
    for i in range(N):
        image = images[i,:,:]
        color = 'red'
        if predictions_str[i] == "Circle":
            color = 'cyan'
        best_cx,best_cy, best_radius = radius_fitted[i,0],radius_fitted[i,1],radius_fitted[i,2]
        circle = Circle((best_cx ,best_cy), best_radius, edgecolor=color, facecolor='none', linewidth=3)
    
      # Plot images
      #for idx in range(N):
        row = i // images_per_row
        col = i % images_per_row
        ax = axes[row, col]
        ax.imshow(final_maps[i],  cmap='inferno')
        #ax.plot(fitimg.x_circle, fitimg.y_circle, 'lime', label=f'Fitted Circle (r={fitimg.best_radius:.2f})', linewidth=1)
        #ax.contour(fitimg.image_padded, levels=[fitimg.threshold], colors='red', linewidths=2)
        ax.add_patch(circle)
    
        ax.set_title(f'{predictions_str[i], i}', fontsize=15,color=color)
        #ax.legend(loc='lower right', fontsize=6, frameon=True)
        ax.axis('off')
    
      # Hide any unused subplots
    for row in range(rows):
        for col in range(cols):
            if row * images_per_row + col >= N:
                axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.savefig(
        f'Mega_Part1_Figures/Diagnostic_Figures/{name}_pred_maps_rsrp{rsrp1}_rsrp{rsrp2}_snr{snr}.png',
        dpi=100,
        bbox_inches='tight',
        pad_inches=0.2
        )
    
    plt.show()
    return
    
snr_values = [10000,500,300,100]

In [ ]:


lc_dir = "/home/iit-t/Gitika/Github-Repositories/Abraham_Mega/Reanalysis_Git/Data_MegaPart1/"  
#snr_values = [10000,500,100]


N = 99
rsrp1 = 10
rsrp2 = 12
maps_folder_str = "10"

obj = MLPreProcessing(Num=Num,N=N,rsrp1=rsrp1,rsrp2=rsrp2,
                              train_frac=train_frac,seed=seed, maps_folder_str = maps_folder_str, test=test, fresh_run = fresh_run)
plot_maps_n_circle_fit(data_dir=lc_dir,N=N,snr=snr,
                       rsrp1=rsrp1,rsrp2=rsrp2
                          )